# 课程项目：rnaseqcount——RNA-seq Count Matrix 分析包
demo名称：rnaseqcount 的功能演示、实战案例与交叉比对  
内容简介：rnaseqcount 是一个面向 bulk RNA-seq count matrix 的轻量化工具包，能够实现过滤、标准化、探索性分析和简化差异表达分析。本demo结合实际数据，展示了 rnaseqcount 的具体工作流程，涵盖数据预处理、过滤低表达基因、标准化、差异表达分析、可视化报告生成的全流程，并整合至工作流，可以对原始数据进行便捷分析  
学习目标：理解 RNA-seq 基本原理，了解原始 gene count matrix 标准化的意义与方法，掌握工具包安装与环境配置，熟练修改代码适配自定义需求  
资源包含：一个完整的Jupyter Notebook代码与分析流程文档，一个完善的 rnaseqcount 工具包（已上传至 Github），一份从GEO数据库上下载的原始 gene count matrix 与 metadata 以及后续分析的中间文件用于demo  

技术路线：  
rnaseqcount 工具包技术路线  
│  
├── 数据准备阶段  
│   ├── 模拟数据生成 (generate_test_data_fixed)  
│   │   └── 输出: counts, metadata, length_dict  
│   └── 真实数据准备 (prepare_from_user_files)  
│       ├── 输入: 原始计数矩阵 + 样本元数据  
│       ├── 处理: 解析、验证、获取基因长度 (ensembl / mygene / 本地)  
│       └── 输出: counts.csv, metadata.csv, length.csv (可选)  
│  
├── 工作流执行 (run_workflow)  
│   ├── 读取标准输入文件 → 构建 CountData 对象  
│   ├── 标准化与过滤  
│   │   ├── TPM / FPKM / CPM  
│   │   └── 低表达基因过滤  
│   ├── 样本可视化  
│   │   ├── PCA 图 (plot_pca)  
│   │   └── 样本相关性热图 (plot_sample_correlation)  
│   ├── 差异表达分析 (differential_test)  
│   │   ├── 分组依据 (grouped_by)  
│   │   ├── t检验 / Wilcoxon (可选配对)  
│   │   └── FDR 多重假设校正  
│   ├── 显著基因筛选 (pick_significance)  
│   └── 火山图绘制 (plot_differential_test)  
│  
├── 输出结果  
│   ├── 差异表达结果表 (CSV)  
│   ├── 显著基因表 (CSV)  
│   └── 图形文件 (PNG)  
│  
└── 可选扩展  
    ├── 手动指定两组比较 (groups 参数)  
    ├── 自定义图形参数  
    └── 集成批次校正等额外步骤  

# 以下是demo的具体展示内容

# 0.从Github上下载rnaseqcount

# 1.rnaseqcount工具包功能展示（以模拟数据为例）

1.1.测试数据生成函数

In [ ]:
def generate_test_data_fixed(
    n_genes: int = 1000,
    n_samples_per_group: int = 5,
    n_groups: int = 2,
    n_batches: int = 2,
    de_ratio: float = 0.3,  # 差异基因比例
    seed: int = 42
) -> tuple:
    """
    修正版：支持多组之间都有差异
    """
    np.random.seed(seed)
    
    # 1. 生成基因名称和长度
    gene_names = [f"gene_{i+1}" for i in range(n_genes)]
    gene_lengths = np.random.lognormal(mean=8, sigma=0.8, size=n_genes)
    gene_lengths = np.round(gene_lengths).astype(int)
    gene_lengths = np.clip(gene_lengths, 500, 50000)
    length_dict = {gene: length for gene, length in zip(gene_names, gene_lengths)}
    
    # 2. 生成样本元数据
    group_names = [f"group_{i+1}" for i in range(n_groups)]
    batch_names = [f"batch_{i+1}" for i in range(n_batches)]
    
    sample_names = []
    group_assignments = []
    batch_assignments = []
    sex_assignments = []
    
    for g_idx, group in enumerate(group_names):
        for s_idx in range(n_samples_per_group):
            sample_name = f"{group}_sample_{s_idx+1}"
            sample_names.append(sample_name)
            group_assignments.append(group)
            batch = np.random.choice(batch_names)
            batch_assignments.append(batch)
            sex = np.random.choice(['M', 'F'])
            sex_assignments.append(sex)
    
    metadata = pd.DataFrame({
        'condition': group_assignments,
        'batch': batch_assignments,
        'sex': sex_assignments
    }, index=sample_names)
    
    # 3. 生成计数矩阵
    counts_matrix = np.zeros((n_genes, len(sample_names)))
    
    # 基础表达水平
    base_expr = np.random.lognormal(mean=5, sigma=1.5, size=n_genes)
    base_expr = base_expr / base_expr.mean() * 1000
    
    # 为每个基因随机选择一个"基准组"（作为参照）
    # 其他组相对于基准组可能有差异
    for g_idx in range(n_genes):
        # 随机决定这个基因是否差异表达
        is_de = np.random.random() < de_ratio
        
        for s_idx, sample in enumerate(sample_names):
            group = group_assignments[s_idx]
            batch = batch_assignments[s_idx]
            
            # 基础表达量
            expr_mean = base_expr[g_idx]
            
            # 如果这个基因是差异表达基因，且不是基准组
            if is_de:
                # 随机选择一个组作为"处理组"（可以是任意组）
                # 这里简化：让 group_2 和 group_3 相对于 group_1 有差异
                if group == group_names[1]:  # group_2
                    fold_change = np.random.choice([-1, 1]) * np.random.uniform(1, 3.3)
                    expr_mean = expr_mean * (2 ** fold_change)
                elif n_groups > 2 and group == group_names[2]:  # group_3
                    fold_change = np.random.choice([-1, 1]) * np.random.uniform(1, 2.5)
                    expr_mean = expr_mean * (2 ** fold_change)
            
            # 批次效应
            if batch == batch_names[1]:
                expr_mean = expr_mean * np.random.uniform(0.8, 1.2)
            
            # 生成计数
            dispersion = 0.2
            if expr_mean > 0:
                overdispersion = np.random.gamma(shape=max(0.1, expr_mean/dispersion), 
                                                scale=dispersion)
                count = np.random.poisson(overdispersion)
            else:
                count = 0
            
            counts_matrix[g_idx, s_idx] = max(0, int(count))
    
    # 转换为DataFrame
    counts = pd.DataFrame(
        counts_matrix,
        index=gene_names,
        columns=sample_names
    )
    counts = counts.astype(int)
    
    # 添加低表达基因
    low_expr_genes = np.random.choice(n_genes, size=int(n_genes*0.1), replace=False)
    for g_idx in low_expr_genes:
        zero_samples = np.random.choice(len(sample_names), 
                                       size=int(len(sample_names)*0.8), 
                                       replace=False)
        for s_idx in zero_samples:
            counts.iloc[g_idx, s_idx] = np.random.poisson(2)
    
    return counts, metadata, length_dict

模拟数据的生成直接给出了counts, metadata, length_dict三个关键量，跳过了从raw_data的预处理环节。  
实际上处理raw_data时，并不会提供gene_length，而是仅提供Ensembl或GeneSymbol格式的gene_id。  
所以length_dict需要通过线上工具，依赖counts_matrix的gene_id获得。  
在真实数据处理中，依赖mygene环境，使用rnaseqcount_prepare获取length_dict。

1.2.统计上述函数生成的测试数据

1.2.1.导入依赖与工具包

In [ ]:
#导入外部依赖
import pandas as pd
import numpy as np
import tempfile
import os
from pathlib import Path
from rnaseqcount import pick_significance
#导入函数模块
from rnaseqcount import (
    CountData, writein, writeout,
    calculate_TPM, filter_low_expression,
    pca, sample_correlation,
    differential_test, plot_pca,
    plot_sample_correlation, plot_differential_test)

1.2.2.生成测试数据

In [ ]:
# 1. 生成测试数据
print("生成模拟 RNA-seq 数据...")
counts, metadata, length_dict = generate_test_data_fixed(
    n_genes=2000,           # 2000个基因
    n_samples_per_group=6,  # 每组6个样本
    n_groups=2,             # 两组：group_1, group_2
    n_batches=2,            # 两个批次
    de_ratio=0.2,           # 20%的基因差异表达
    seed=42
)
print(f"计数矩阵形状: {counts.shape}")
print(f"元数据形状: {metadata.shape}")
print(f"基因长度数量: {len(length_dict)}")

1.2.3.构建 CountData 对象

In [ ]:
# 2. 构建 CountData 对象
print("\n构建 CountData 对象...")
cdata = CountData(counts=counts, metadata=metadata, length=length_dict)
print(f"基因数: {cdata.n_genes}, 样本数: {cdata.n_samples}")

1.2.4.TPM 标准化并过滤低表达基因

In [ ]:
# 3. TPM 标准化并过滤低表达基因（TPM > 1 的基因至少在一个样本中）
print("\n执行 TPM 标准化和低表达过滤...")
tpm_expr = calculate_TPM(cdata)
filtered_expr = filter_low_expression(cdata, method="TPM", filter_threshold=1.0)
print(f"过滤后基因数: {filtered_expr.shape[0]} / 原始 {cdata.n_genes}")

1.2.5.PCA 分析并绘图

In [ ]:
# 4. PCA 分析并绘图（结果保存至临时目录）
out_dir = tempfile.mkdtemp()
print(f"输出目录: {out_dir}")
print("\n进行 PCA 分析...")
fig_pca = plot_pca(
    cdata, method="TPM", filter_threshold=1.0,
    label_name="condition",  # 用 condition 列标记样本颜色
    show_labels=True,
    figsize=(10, 8)
)
pca_output = os.path.join(out_dir, "pca_plot")
writeout(fig_pca, pca_output)

1.2.6.绘制样本相关性热图

In [ ]:
# 5. 样本相关性热图
print("\n计算样本相关性并绘图...")
fig_corr = plot_sample_correlation(
    cdata, method="TPM", filter_threshold=1.0,
    cmap='coolwarm'
)
corr_output = os.path.join(out_dir, "sample_correlation")
writeout(fig_corr, corr_output)

1.2.7.差异表达分析

In [ ]:
# 6. 差异表达分析：比较 group_1 vs group_2
print("\n执行差异表达分析 (group_1 vs group_2, t检验 + FDR)...")
diff_res = differential_test(
    cdata,
    method="TPM",
    filter_threshold=1.0,
    grouped_by="condition",        # 使用 metadata 中的 condition 列
    groups=None,                   # 自动使用该列的所有值（这里应为 group_1, group_2）
    log2_transform=True,
    fdr_correction=True,
    test_method="t_test",
    paired=False
)
# 保存完整差异表达结果
diff_output = os.path.join(out_dir, "diff_results")
writeout(diff_res, diff_output)
print(f"差异表达结果前5行:\n{diff_res.head()}")

1.2.8.提取显著差异基因

In [ ]:
# 7. 提取显著差异基因（FDR < 0.05）
sig_genes = pick_significance(
    cdata,
    method="TPM",
    filter_threshold=1.0,
    grouped_by="condition",
    fdr_correction=True,
    significance_threshold=0.05
)
sig_output = os.path.join(out_dir, "significant_genes")
writeout(sig_genes, sig_output)
print(f"显著差异基因数量 (FDR<0.05): {len(sig_genes)}")

1.2.9.绘制火山图

In [ ]:
# 8. 绘制火山图
print("\n绘制火山图...")
fig_volcano = plot_differential_test(
    cdata,
    method="TPM",
    filter_threshold=1.0,
    grouped_by="condition",
    fdr_correction=True,
    test_method="t_test",
    significance_threshold=0.05,
    log2fc_threshold=1.0     # |log2FC| > 1 视为生物意义显著
)
volcano_output = os.path.join(out_dir, "volcano_plot")
writeout(fig_volcano, volcano_output)
    
print(f"\n所有输出文件已保存至: {out_dir}")
print("分析完成！")

# 2.默认workflow演示

In [ ]:
from rnaseqcount_wf import run_workflow
run_workflow(
    data_source='simulated', # 默认为real，若为real，则需指定输入文件路径（gene count matrix，metadata以及lengths）
    n_genes=500,
    n_samples_per_group=4,
    n_groups=2,
    de_ratio=0.2,
    method='TPM', # 可选TPM,FPKM,CPM
    filter_threshold=1.0,
    grouped_by='condition', # 默认为condition，（即metadata中的第二列列名，表示对组别的处理条件，一般为control和treatment）
    test_method='t_test',
    fdr_correction=True,
    significance_threshold=0.05,
    log2fc_threshold=1.0,
    output_dir='./rnaseq_demo/simulated',
    prefix='simulated_demo'
)

# 3.实际数据测试

3.1.从GEO数据库上下载并处理原始数据

3.1.1.下载 GSE316117 的GSE316117_counts_allsamples.txt.gz文件（原始的gene count），以及SraRunTable.csv文件（metadata文件），保存至./real_sample_for_demo  
GSE316117是


3.1.2.使用rnaseqcount_prepare包对下载的真实数据作预处理，获得格式标准化、可直接用于下一步分析的counts, metadata, length_dict文件

In [ ]:
from rnaseqcount_prepare import prepare_from_user_files

result = prepare_from_user_files(
    metadata_path="./real_sample_for_demo/SraRunTable.csv",
    counts_path="./real_sample_for_demo/GSE316117_counts_allsamples.txt.gz",
    output_dir="./rnaseq_demo/real_prepare",
    length_source='mygene',   # 使用 mygene 查询基因符号长度(可选ensembl),需网络
    species='hsapiens',  # 指定物种为homo sapiens
    condition_col="treatment",  # 使用 treatment 列作为分组条件
    sample_id_col="sample_name",  # 指定样本名列
)

print(result)

当然，也可以手动配置metadata文件  
若有必要，需调整输出的metadata中treatment列下key出现的顺序，以确保实验组和对照组的相对位置（对火山图作图有影响，默认treatment列下第一个出现的key为实验组，而第二个出现的key作为基准对照）

3.2.利用rnaseqcount_wf包处理数据

In [ ]:
from rnaseqcount_wf import run_workflow
run_workflow(
    data_source ='real',
    input_dir="./rnaseq_demo/real_prepare",  # 即rnaseqcount_prepare输出的预处理结果输出路径
    method='TPM',
    filter_threshold=1.0,
    test_method='t_test',
    fdr_correction=True,
    significance_threshold=0.05,
    log2fc_threshold=1.0,
    output_dir='./rnaseq_demo/real_output',
    prefix='real_demo'
)

运行，成功输出样本相关性热图、PCA散点图、火山图

3.3.与其他工具包进行交叉比对

使用DEseq2对GSE316117_counts_allsamples.txt文件进行差异表达分析，使用参数：低表达量过滤阈值=1.0  
DEseq2的输出结果见".\rnaseq_demo\DEseq2_ouput\DEseq2_ouput.csv"  
之后使用参数：pvalue阈值=0.05，log2(Fold Change)阈值=1.0作火山图  
使用代码及输出火山图同见".\rnaseq_demo\DEseq2_ouput\"  

发现rnaseqcount与DEseq2输出的火山图见有明显差异，推测是由于统计模型不同导致的。DEseq2使用负二项广义线性模型（GLM），而在本demo中，rnaseqcount使用t检验，导致火山图的 -log10(p) 分布不同。但总体来说，都输出了具有比较意义的火山图。